# 📝 Universal JSON Flattener
This Colab notebook flattens nested JSONs into a clean tabular DataFrame.

## 🔼 Upload Your JSON Files

In [ ]:
from google.colab import files
uploaded = files.upload()

import json

# Load all uploaded JSONs into a list
json_data = []
for fname in uploaded:
    with open(fname, 'r') as f:
        json_data.append(json.load(f))
print(f"Loaded {len(json_data)} JSON records.")


## 🛠️ Reusable Function: `flatten_json_records`

In [ ]:
import pandas as pd

def flatten_json_records(json_list, max_items=5):
    """
    Flattens a list of JSON dicts into a Pandas DataFrame.
    Handles nested dicts and lists of dicts (to a fixed max length).

    Parameters:
        json_list: List[dict] - list of JSON objects
        max_items: int - max number of items to flatten for any list of dicts

    Returns:
        pd.DataFrame - flattened tabular DataFrame
    """

    # Step 1: Flatten nested dicts using json_normalize
    df = pd.json_normalize(json_list, sep='.')

    # Step 2: Identify list-of-dicts columns
    list_cols = [col for col in df.columns if isinstance(df[col].dropna().iloc[0], list)]

    for col in list_cols:
        # Detect max number of elements for that column
        max_len = max(df[col].apply(lambda x: len(x) if isinstance(x, list) else 0))
        max_len = min(max_len, max_items)

        for i in range(max_len):
            df[f"{col}[{i}]"] = df[col].apply(
                lambda x: x[i] if isinstance(x, list) and len(x) > i else None
            )
            # If it's a dict, unpack it
            if df[f"{col}[{i}]"].dropna().apply(lambda v: isinstance(v, dict)).any():
                sub_df = pd.json_normalize(df[f"{col}[{i}]"], sep='.')
                sub_df.columns = [f"{col}[{i}].{sub}" for sub in sub_df.columns]
                df = pd.concat([df, sub_df], axis=1)
                df.drop(columns=[f"{col}[{i}]"], inplace=True)
        df.drop(columns=[col], inplace=True)

    return df


## 📊 Flatten and Inspect

In [ ]:
df_flat = flatten_json_records(json_data, max_items=5)
print("✅ Flattened shape:", df_flat.shape)
df_flat.head()


## 💾 Download the Flattened Data

In [ ]:
df_flat.to_csv("flattened_output.csv", index=False)
files.download("flattened_output.csv")
